# Hipótesis HB1: satisfacción laboral y postura hacia la IA

> HB1) La satisfacción laboral (`JobSat`) se relaciona con la postura hacia el uso de herramientas de IA (`AISent`).

Esta hipótesis es **bivariada** (dos variables) y usa el **Subgrupo 2**. Seguimos el flujo de la práctica 6 de la cátedra: gráfico, normalidad, homocedasticidad, elección de la prueba y lectura con alfa = 0,05.

In [ ]:
%pip install pandas seaborn matplotlib scipy wget
# Si es la primera vez que se instalan, reiniciar el kernel (Kernel -> Restart)

In [ ]:
from wget import download
from os import path, listdir
import pandas as pd
# PARA AUMENTAR EL ANCHO PERMITIDO Y Q NO VEA DATOS CORTADOS
pd.set_option('display.max_colwidth', None)
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import shapiro, kstest

In [ ]:
if not path.exists("schema.csv"):
  download("https://media.githubusercontent.com/media/StackExchange/Survey/refs/heads/main/packages/archive/2025/schema.csv")
if not path.exists("results.csv"):
  download("https://media.githubusercontent.com/media/StackExchange/Survey/6ae2209cd35a55e92739ba82f7887fb86807a069/packages/archive/2025/results.csv")

In [ ]:
# low_memory=False evita el DtypeWarning de las columnas con tipos mezclados
raw_dataset_results = pd.read_csv("results.csv", low_memory=False)
raw_dataset_schema = pd.read_csv("schema.csv")

In [ ]:
# 1. Definición de las listas de columnas para cada subgrupo
cols_g1 = ['Country', 'WorkExp', 'ConvertedCompYearly', 'CompTotal', 'Currency', 'Age', 'EdLevel', 'Employment', 'YearsCode']
cols_g2 = ['WorkExp', 'AIAcc', 'JobSat', 'AISent', 'AISelect', 'AIThreat', 'AILearnHow']

# Para el Grupo 3 extraemos dinámicamente todas las de JobSatPoints (del 1 al 15 + TEXT) para evitar hardcodearlas
cols_jobsat = [col for col in raw_dataset_results.columns if col.startswith('JobSatPoints')]
cols_g3 = ['NewRole', 'JobSat'] + cols_jobsat

cols_g4 = ['ConvertedCompYearly', 'WorkExp', 'OrgSize', 'AISelect', 'RemoteWork', 'DevType', 'Industry', 'ICorPM']

# 2. Creación de los DataFrames independientes para cada subgrupo
# Usamos .copy() para que cada compañero pueda limpiar nulos o transformar datos sin afectar el dataset original

# Subgrupo 1: Perfil Económico (Para Hipótesis A y HU1)
df_perfil_economico = raw_dataset_results[cols_g1].copy()

# Subgrupo 2: Impacto de la IA (Para Hipótesis B y HB1)
df_impacto_ia = raw_dataset_results[cols_g2].copy()

# Subgrupo 3: Cambio de Rol (Para Hipótesis C)
df_cambio_rol = raw_dataset_results[cols_g3].copy()

# Subgrupo 4: Premio Salarial (Para Hipótesis M4)
df_premio_salarial = raw_dataset_results[cols_g4].copy()

Control: las columnas de cada subgrupo existen en `results` (cada lista debe salir vacía) y el esquema describe las del subgrupo 2.

In [ ]:
for nombre, cols in {"g1": cols_g1, "g2": cols_g2, "g3": cols_g3, "g4": cols_g4}.items():
    print(nombre, [c for c in cols if c not in raw_dataset_results.columns])

raw_dataset_schema[raw_dataset_schema["qname"].isin(cols_g2)][["qid", "qname", "type", "question"]]

Transformacion sobre Subgrupo 2: Satisfaccion laboral vs Postura hacia la IA

In [ ]:
# 1. Chequeo del rango de JobSat (escala 0 a 10)
print("JobSat fuera de 0-10:", ((df_impacto_ia["JobSat"] < 0) | (df_impacto_ia["JobSat"] > 10)).sum())
df_impacto_ia["JobSat"].value_counts(dropna=False).sort_index()

Justificacion de limpieza para el informe:

- **Sin valores fuera de rango en `JobSat`:** es una escala de 0 a 10 y se comprobó que ningún valor la excede, así que no hizo falta corregirla. Los valores bajos (0, 1 y 2) son personas muy insatisfechas: son respuestas reales y **no se eliminan**.
- **Rechazo a la imputación de datos:** `JobSat` tiene cerca de la mitad de nulos y `AISent` cerca de un tercio. La pregunta de `JobSat` habla del "rol profesional actual como desarrollador", por lo que posiblemente no se mostró a quienes no trabajan (es una hipótesis, no un hecho). Rellenarlos con promedios o medianas introduciría un sesgo artificial en una encuesta de opinión.
- **Estrategia de eliminación segura (Subsetting):** no se aplica un `.dropna()` global sobre las 172 columnas, porque las columnas de texto libre con altísima tasa de nulos eliminarían casi todas las filas válidas. Primero se aíslan las dos variables y recién entonces se eliminan las filas con nulos solo en ellas.
- **Respuesta "Unsure" de `AISent`:** significa "no sé", no es un punto de la escala entre "Indifferent" y otra categoría. Se excluye de esta hipótesis pasándola a nulo (alrededor de 760 personas, 1,5%).
- **Codificación manual de `AISent`:** es ordinal (5 niveles). El orden se define a mano con un diccionario porque pandas ordenaría alfabéticamente.

Validación de Hipótesis HB1

¿Está el nivel de satisfacción laboral relacionado con la postura frente al uso de herramientas de IA?

Partimos del Subgrupo 2 (Impacto de la IA). Las variables que usamos son:

- **Variable 1 (`JobSat`):** numérica discreta de 0 a 10, concentrada en 7 a 9 y con cola hacia los valores bajos.
- **Variable 2 (`AISent`):** categórica ordinal (MC) de 5 niveles, de "Very unfavorable" (1) a "Very favorable" (5).

Exclusión de variables no necesarias: las demás columnas del subgrupo (`WorkExp`, `AIAcc`, `AISelect`, `AIThreat`, `AILearnHow`) no intervienen en esta hipótesis.

Planteo estadístico: **H0:** la satisfacción laboral es la misma en los 5 niveles de postura. **H1:** en al menos un nivel es distinta. Nivel de significación: alfa = 0,05.

In [ ]:
# 1. Codificación ordinal de AISent (1 = muy desfavorable, 5 = muy favorable). "Unsure" queda fuera del diccionario
orden_aisent = {"Very unfavorable": 1, "Unfavorable": 2, "Indifferent": 3,
                "Favorable": 4, "Very favorable": 5}
df_impacto_ia["AISent_num"] = df_impacto_ia["AISent"].map(orden_aisent)

# Control: respuestas no nulas que quedaron sin código (solo debería aparecer "Unsure")
print(df_impacto_ia[df_impacto_ia["AISent"].notna() & df_impacto_ia["AISent_num"].isna()]["AISent"].value_counts())

# 2. Nos quedamos con las dos variables y con las filas que tienen ambas respuestas
df_hipotesis_hb1 = df_impacto_ia[["JobSat", "AISent", "AISent_num"]].dropna().copy()

print(f"Total de observaciones listas para el análisis: {len(df_hipotesis_hb1)}")
df_hipotesis_hb1.head()

Análisis de las variables

Antes de aplicar las pruebas miramos cómo se distribuye cada variable. `AISent` es una variable cualitativa ordinal. `JobSat` toma solo 11 valores enteros y es asimétrica hacia la izquierda.

In [ ]:
print("Personas por nivel de AISent:")
print(df_hipotesis_hb1["AISent_num"].value_counts().sort_index())

print("Asimetría (skew) de JobSat:", round(df_hipotesis_hb1["JobSat"].skew(), 2))

sns.histplot(df_hipotesis_hb1["JobSat"], bins=11, discrete=True)
plt.title("Distribución de JobSat")
plt.show()

Paso 1: Exploración visual

Graficamos la satisfacción de cada nivel de postura con un boxplot con muesca (`notch=True`). Como `JobSat` es discreta, la mediana y el cuartil superior pueden coincidir y el boxplot se ve "plano" en algunos grupos: no es un error del gráfico. Los puntos sueltos en 0, 1 y 2 son personas muy insatisfechas, no se eliminan. Es una pista visual, **no una prueba**.

In [ ]:
print("Media de JobSat por nivel de AISent:")
print(df_hipotesis_hb1.groupby("AISent_num")["JobSat"].mean())
print("Mediana de JobSat por nivel de AISent:")
print(df_hipotesis_hb1.groupby("AISent_num")["JobSat"].median())

plt.figure(figsize=(8, 4.5))
sns.boxplot(x="AISent_num", y="JobSat", data=df_hipotesis_hb1, notch=True)
plt.xlabel("AISent (1 = muy desfavorable, 5 = muy favorable)")
plt.ylabel("JobSat (0-10)")
plt.title("Satisfacción laboral según postura hacia la IA")
plt.show()

Paso 2: Normalidad

Hay que ver si `JobSat` es normal **dentro de cada grupo**. Usamos Shapiro-Wilk, Kolmogorov-Smirnov (estandarizando antes los datos) y QQ plot. La H0 de normalidad dice que los datos siguen una distribución normal: si p < 0,05 se rechaza. Con grupos de más de 5.000 datos `scipy` avisa que el p-valor de Shapiro es aproximado, es esperable.

In [ ]:
grupos_hb1 = [g["JobSat"] for _, g in df_hipotesis_hb1.groupby("AISent_num")]

for i, g in enumerate(grupos_hb1, start=1):
    stat, p = shapiro(g)
    print(f"AISent = {i}: Shapiro-Wilk Estadístico={stat:.3f}, p-valor={p:.3g}")

for i, g in enumerate(grupos_hb1, start=1):
    stat, p = kstest((g - g.mean()) / g.std(), 'norm')
    print(f"AISent = {i}: Kolmogorov-Smirnov Estadístico={stat:.3f}, p-valor={p:.3g}")

In [ ]:
fig, ejes = plt.subplots(1, 5, figsize=(18, 3.5))
for i, g in enumerate(grupos_hb1):
    stats.probplot(g, dist="norm", plot=ejes[i])
    ejes[i].set_title(f"QQ plot AISent = {i + 1}")
plt.tight_layout()
plt.show()

Si los tests rechazan normalidad y el QQ plot se curva, **no se puede usar ANOVA** y corresponde una prueba no paramétrica.

Paso 3: Homocedasticidad (Levene)

H0: las varianzas de los 5 grupos son **iguales**. Si p < 0,05 se rechaza y las varianzas son distintas.

In [ ]:
stat, p = stats.levene(*grupos_hb1)
print(f"Test de Levene para JobSat: Estadístico={stat:.3f}, p-valor={p:.3g}")

alpha = 0.05
if p > alpha:
    print("No hay suficiente evidencia para rechazar la hipótesis nula.")
    print("Las varianzas de JobSat son iguales entre los grupos de AISent.")
else:
    print("Se rechaza la hipótesis nula.")
    print("Las varianzas de JobSat son diferentes entre los grupos de AISent.")

Paso 4: Prueba de hipótesis (Kruskal-Wallis)

Hay **5 grupos independientes** y `JobSat` no es normal. Por eso usamos **Kruskal-Wallis**, la alternativa no paramétrica al ANOVA para comparar más de dos grupos. Solo dice **si hay diferencia** en algún grupo, no entre cuáles ni en qué sentido.

In [ ]:
stat_kw, p_kw = stats.kruskal(*grupos_hb1)
print(f"Test de Kruskal-Wallis para JobSat: Estadístico={stat_kw:.3f}, p-valor={p_kw:.3g}")

alpha = 0.05
if p_kw > alpha:
    print("No hay suficiente evidencia para rechazar la hipótesis nula.")
    print("No hay una diferencia significativa en JobSat entre los niveles de AISent.")
else:
    print("Se rechaza la hipótesis nula.")
    print("Existe una diferencia significativa en JobSat entre los niveles de AISent.")

Paso 5: Sentido de la asociación

Calculamos la correlación entre las dos variables. `corr(method="spearman")` usa rangos, adecuada para una variable ordinal. Va de -1 a 1: positiva significa que a postura más favorable, más satisfacción.

In [ ]:
tabla_corr = df_hipotesis_hb1[["JobSat", "AISent_num"]].corr(method="spearman")
rho = tabla_corr.loc["JobSat", "AISent_num"]
print(tabla_corr)
print(f"\nCorrelación de Spearman = {rho:.3f}")

Resumen de los resultados (se arma solo con los números calculados, para que el texto no pueda contradecirlos). Hay que revisarlo y reescribirlo con palabras propias.

In [ ]:
decision = "se rechaza H0" if p_kw < alpha else "no se rechaza H0"
sentido = "positiva (postura más favorable, más satisfacción)" if rho > 0 else "negativa (postura más favorable, menos satisfacción)"
print(f"n = {len(df_hipotesis_hb1)}")
print(f"Kruskal-Wallis: H = {stat_kw:.2f}, p = {p_kw:.3g} -> con alfa = {alpha}, {decision}.")
if p_kw < alpha:
    print(f"Correlación de Spearman = {rho:.3f} (asociación {sentido}).")
else:
    print(f"Correlación de Spearman = {rho:.3f}. Como no se rechaza H0, no se interpreta un sentido de la asociación.")

Conclusión Final de la Hipótesis HB1

*(Completar con los números del resumen anterior: p-valor de Kruskal-Wallis, medias y medianas por grupo y valor de la correlación; decir si se rechaza H0 y en qué sentido va la asociación.)*

Cuidados al interpretar:

1. **Significativo no es importante.** Con unas 24.000 filas, una diferencia de medio punto en una escala de 0 a 10 puede dar un p-valor minúsculo y ser chica en la práctica. Hay que mirar el tamaño de la diferencia.
2. **No rechazar H0 no prueba que H0 sea cierta.**
3. **Asociación no es causalidad:** la encuesta es observacional.

Limitaciones: encuesta voluntaria y online; muchos nulos (`JobSat` cerca de la mitad, `AISent` cerca de un tercio) sin imputar; se excluyó "Unsure"; las escalas se codificaron sin asumir niveles equidistantes; Kruskal-Wallis no indica qué grupos difieren entre sí.